# LLM on Azure

In this notebook we look at how to use an LLM on Azure AI Foundry. We use this intead of Ollama to gain access to larger LLMs. It is based on the tutorial at [learn.microsoft.com](https://learn.microsoft.com/en-us/azure/ai-foundry/quickstarts/get-started-code?tabs=python). It is also important, that we save the answers after asking them, such that we don't query the model multiple times (which costs money).

We need another python library (already handled in `env.yml`)

In [16]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


We load the secrets from the `.env` file.

In [17]:
from dotenv import Dotenv

env = Dotenv(".env")
AZURE_KEY = env.get("AZURE_KEY")
AZURE_ENDPOINT = env.get("AZURE_ENDPOINT")

In [ ]:
import os

import pandas as pd
from openai import AzureOpenAI

deployment = "gpt-5-nano"
api_version = "2024-12-01-preview"

CACHE_DIR = "data"
CACHE_FILE = os.path.join(CACHE_DIR, "azure_ai_cache.pkl")

os.makedirs(CACHE_DIR, exist_ok=True)

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_KEY,
)


def load_cache():
    if os.path.exists(CACHE_FILE):
        return pd.read_pickle(CACHE_FILE)
    return pd.DataFrame(columns=["question", "answer"])


def save_cache(df):
    df.to_pickle(CACHE_FILE)


def ask_model(question: str, system_prompt: str = "You are a helpful assistant."):
    df = load_cache()

    cached = df.loc[df["question"] == question]
    if not cached.empty:
        print("Loaded from cache")
        return cached.iloc[0]["answer"]

    print("Querying Azure OpenAI...")
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        max_completion_tokens=16384,
    )

    answer = response.choices[0].message.content

    new_row = pd.DataFrame([{"question": question, "answer": answer}])
    df = pd.concat([df, new_row], ignore_index=True)
    save_cache(df)

    return answer


question = "I am going to Paris, what should I see?"
answer = ask_model(question)
print("\nAnswer:\n", answer, sep="")

Loaded from cache

Answer:
Fantastic choice—Paris is magical. To tailor suggestions, tell me a bit about how many days you’ll have, your interests (art, food, architecture, shopping, nightlife), and your budget. In the meantime, here’s a practical starter guide with must-sees, neighborhoods, and a couple of sample itineraries.

Top must-sees (iconic highlights)
- Eiffel Tower and Trocadéro view: book a timed-entry or summit ticket to skip long lines.
- Louvre Museum: plan 2–3 hours for a focused visit (or pick a few key works like the Pyramid, Mona Lisa, Venus de Milo). Book tickets in advance.
- Île de la Cité: Notre-Dame area and Sainte-Chapelle (stunning stained glass).
- Montmartre and Sacré‑Cœur: bohemian vibes, great Parisian views from the steps.
- Seine river cruise: a relaxing way to see many bridges and landmarks, especially nice at sunset.
- Musée d'Orsay (impressionist masterpieces) and/or Centre Pompidou (modern art) for a taste of different eras.
- Jardin du Luxembourg or